# 07 Shift And Confounding Review

This notebook supports the report section on reliability: quality sensitivity, subgroup checks, and signs that the model may be learning dataset or demographic shortcuts.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
require_artifacts(["data/processed/metadata_clean.csv"])
metadata = read_csv("data/processed/metadata_clean.csv", n=5)
subgroup = read_optional_csv("data/outputs/metrics/subgroup_metrics.csv")
fusion_pred = read_optional_csv("data/outputs/metrics/fusion_predictions.csv")
branch_pred = read_optional_csv("data/outputs/metrics/calibrated_branch_predictions.csv")


## Metadata Coverage


In [ ]:
demographic_cols = [c for c in ["age", "gender", "country", "symptoms_json", "comorbidities_json", "quality_flag", "recording_date"] if c in metadata.columns]
coverage = pd.DataFrame({
    "column": demographic_cols,
    "non_empty_rate": [metadata[c].replace("", np.nan).notna().mean() for c in demographic_cols],
    "n_unique": [metadata[c].nunique(dropna=True) for c in demographic_cols],
})
save_table(coverage, "reports/tables/nb07_metadata_coverage.csv")
display(coverage)


## Subgroup Metrics


In [ ]:
if subgroup is not None and not subgroup.empty:
    metric_cols = [c for c in ["analysis_group", "quality_flag", "gender", "age_bucket", "auroc", "auprc", "balanced_accuracy", "sensitivity", "specificity", "ece", "n_samples"] if c in subgroup.columns]
    save_table(subgroup[metric_cols], "reports/tables/nb07_subgroup_metrics.csv")
    display(subgroup[metric_cols])
    if "n_samples" in subgroup.columns and (subgroup["n_samples"] < 20).any():
        print("Warning: some subgroup metrics have n<20 and should be treated as descriptive only.")
else:
    print("No subgroup metrics found. Run scripts/10_shift_and_confounding_checks.py after fusion.")


## Quality Sensitivity


In [ ]:
predictions = branch_pred if branch_pred is not None and "recording_id" in branch_pred.columns else fusion_pred
if predictions is not None:
    if "recording_id" in predictions.columns:
        merge_cols = [c for c in ["recording_id", "participant_id", "quality_flag", "modality", "label_binary"] if c in metadata.columns]
        merged = predictions.merge(metadata[merge_cols], on="recording_id", how="left", suffixes=("", "_meta"))
    elif "participant_id" in predictions.columns:
        participant_quality = metadata.groupby("participant_id").agg(
            quality_flag=("quality_flag", lambda x: x.value_counts(dropna=False).index[0] if len(x) else "unknown")
        ).reset_index()
        merged = predictions.merge(participant_quality, on="participant_id", how="left")
    else:
        merged = pd.DataFrame()

    if not merged.empty and "quality_flag" in merged.columns:
        quality_prediction_summary = merged.groupby("quality_flag").agg(
            n=("probability", "count"),
            mean_probability=("probability", "mean"),
            median_probability=("probability", "median"),
        ).reset_index()
        save_table(quality_prediction_summary, "reports/tables/nb07_quality_prediction_summary.csv")
        display(quality_prediction_summary)
        sns.boxplot(data=merged, x="quality_flag", y="probability")
        plt.xticks(rotation=25)
        plt.title("Predicted probability by quality flag")
        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / "reports/figures/nb07_quality_sensitivity.png", dpi=160)
        plt.show()
else:
    print("No prediction artifact found for quality sensitivity.")


## Recording Date And Dataset Shift Readiness


In [ ]:
if "recording_date" in metadata.columns:
    dates = pd.to_datetime(metadata["recording_date"], errors="coerce")
    date_summary = pd.DataFrame({
        "n_with_date": [dates.notna().sum()],
        "n_total": [len(metadata)],
        "min_date": [dates.min()],
        "max_date": [dates.max()],
    })
    save_table(date_summary, "reports/tables/nb07_recording_date_summary.csv")
    display(date_summary)
    if dates.notna().mean() < 0.50:
        print("Recording dates are too sparse for strong temporal drift claims. Treat drift analysis as limited.")
else:
    print("No recording_date column. Temporal drift analysis cannot be claimed strongly.")


## Report Decision Notes


In [ ]:
notes = []
if subgroup is None or subgroup.empty:
    notes.append("Subgroup metrics missing; do not claim demographic robustness.")
if "gender" in metadata.columns and metadata["gender"].replace("", np.nan).notna().mean() < 0.50:
    notes.append("Gender coverage is sparse; subgroup fairness claims are descriptive only.")
if "quality_flag" in metadata.columns and (metadata["quality_flag"] != "ok").mean() > 0.20:
    notes.append("More than 20% recordings are non-ok quality; include quality-filtered sensitivity results.")
if not notes:
    notes.append("No automatic warning triggered. Still present these checks as reliability limitations, not clinical validation.")
report_notes = pd.DataFrame({"note": notes})
save_table(report_notes, "reports/tables/nb07_report_decision_notes.csv")
display(report_notes)
